# Black Hole Simulation (Stargate Element)

This notebook simulates black holes of various masses, from micro (Hawking radiation) to supermassive. Built for the Cosmic Engine project.

In [ ]:
!pip install taichi numpy matplotlib pillow

In [ ]:
import os
os.makedirs("physics", exist_ok=True)
os.makedirs("render", exist_ok=True)

with open("physics/black_hole.py", "w") as f: f.write("""
import taichi as ti
import math

@ti.data_oriented
class BlackHole:
    def __init__(self):
        pass

    @ti.func
    def get_disk_color(self, r, time, mass, disk_inner, disk_outer):
        normalized_r = (r - disk_inner) / (disk_outer - disk_inner + 1e-6)
        temp = (1.0 - normalized_r) * (2.0 / (mass**0.5 + 0.5))
        color = ti.Vector([0.0, 0.0, 0.0])
        if mass < 0.5:
            color = ti.Vector([0.5, 0.8, 1.0]) * temp * 2.0
        elif mass < 5.0:
            color = ti.Vector([1.0, 0.6, 0.2]) * temp
        else:
            color = ti.Vector([1.0, 0.2, 0.05]) * temp * 0.5
        noise = ti.sin(r * 10.0 - time * 5.0 / (mass + 0.1)) * 0.2 + 0.8
        return color * noise

    @ti.func
    def render(self, ro, rd, time, mass):
        rs = 2.0 * mass
        r_isco = 3.0 * rs
        disk_inner = r_isco
        disk_outer = r_isco * 4.0
        hawking_temp = 1.0 / (mass + 1e-6)
        curr_p = ro
        curr_v = rd
        total_color = ti.Vector([0.0, 0.0, 0.0])
        transmittance = 1.0
        dt = 0.1
        max_steps = 100
        for i in range(max_steps):
            r2 = curr_p[0]**2 + curr_p[1]**2 + curr_p[2]**2
            r = ti.sqrt(r2)
            if r < rs * 1.01:
                transmittance = 0.0
                break
            if r > 15.0 * mass + 5.0:
                break
            force_mag = (1.5 * rs) / (r2 * r + 1e-6)
            accel = -curr_p * force_mag
            curr_v += accel * dt
            curr_v /= ti.sqrt(curr_v[0]**2 + curr_v[1]**2 + curr_v[2]**2)
            curr_p += curr_v * dt
            if ti.abs(curr_p[1]) < 0.05:
                dist_xz = ti.sqrt(curr_p[0]**2 + curr_p[2]**2)
                if disk_inner < dist_xz < disk_outer:
                    disk_col = self.get_disk_color(dist_xz, time, mass, disk_inner, disk_outer)
                    velocity_dot_view = -curr_p[2] / (dist_xz + 1e-6)
                    beaming = 1.0 + velocity_dot_view * 0.8
                    total_color += disk_col * beaming * transmittance
                    transmittance *= 0.1
            if transmittance < 0.01:
                break
        if mass < 0.1:
            hawking = ti.Vector([0.8, 0.9, 1.0]) * hawking_temp * 0.01
            sparkle = ti.random() * ti.exp(- (ti.sqrt(ro[0]**2 + ro[1]**2 + ro[2]**2) - rs)**2 * 10.0)
            total_color += hawking * sparkle
        return total_color
""")
with open("render/geodesic_engine.py", "w") as f: f.write("""
import taichi as ti
import math
from physics.black_hole import BlackHole

@ti.data_oriented
class GeodesicEngine:
    def __init__(self, render_res=1024, samples=4):
        self.render_res = render_res
        self.samples = samples
        self.final_output = ti.Vector.field(3, dtype=float, shape=(self.render_res, self.render_res))
        self.observer_attention = ti.field(dtype=float, shape=())
        self.observer_attention[None] = 1.0 # Force attention for Colab demo
        self.bh_mass = ti.field(dtype=float, shape=())
        self.bh_mass[None] = 1.0
        self.black_hole = BlackHole()

    @ti.kernel
    def update_black_hole_mass(self, mass: float):
        self.bh_mass[None] = mass
""")
with open("render/camera.py", "w") as f: f.write("""
import taichi as ti
import math
import numpy as np
from render.geodesic_engine import GeodesicEngine

@ti.data_oriented
class SphereCamera:
    def __init__(self, render_res=1024, samples=4):
        self.render_res = render_res
        self.samples = samples
        self.geo_engine = GeodesicEngine(render_res=render_res, samples=samples)
        self.cam_tilt = -0.3
        self.cam_pan = 0.0
        self.cam_roll = 0.0
        self.final_output = ti.Vector.field(3, dtype=float, shape=(self.render_res, self.render_res))

    @ti.func
    def rot_x(self, v, angle):
        c, s = ti.cos(angle), ti.sin(angle)
        return ti.Vector([v[0], v[1]*c - v[2]*s, v[1]*s + v[2]*c])
    @ti.func
    def rot_y(self, v, angle):
        c, s = ti.cos(angle), ti.sin(angle)
        return ti.Vector([v[0]*c + v[2]*s, v[1], -v[0]*s + v[2]*c])
    @ti.func
    def rot_z(self, v, angle):
        c, s = ti.cos(angle), ti.sin(angle)
        return ti.Vector([v[0]*c - v[1]*s, v[0]*s + v[1]*c, v[2]])

    @ti.kernel
    def render_black_hole(self, time: float):
        mass = self.geo_engine.bh_mass[None]
        for i, j in self.final_output:
            accumulated_color = ti.Vector([0.0, 0.0, 0.0])
            for k in range(self.samples):
                u = (float(i) + ti.random() - 0.5) / self.render_res * 2.0 - 1.0
                v = (float(j) + ti.random() - 0.5) / self.render_res * 2.0 - 1.0
                ro = ti.Vector([0.0, 0.0, -8.0])
                rd = ti.Vector([u, v, 1.5])
                rd /= ti.sqrt(rd[0]**2 + rd[1]**2 + rd[2]**2)
                rd = self.rot_z(rd, self.cam_roll)
                ro = self.rot_x(ro, self.cam_tilt); rd = self.rot_x(rd, self.cam_tilt)
                ro = self.rot_y(ro, self.cam_pan); rd = self.rot_y(rd, self.cam_pan)
                accumulated_color += self.geo_engine.black_hole.render(ro, rd, time, mass)
            final_color = accumulated_color / float(self.samples)
            a, b, c, d, e = 2.51, 0.03, 2.43, 0.59, 0.14
            color = (final_color * (a * final_color + b)) / (final_color * (c * final_color + d) + e)
            self.final_output[i, j] = color

    def get_image_data(self):
        return np.clip(self.final_output.to_numpy(), 0.0, 1.0)
""")

In [ ]:
import taichi as ti
import numpy as np
import matplotlib.pyplot as plt
import time
import math
from render.camera import SphereCamera
from IPython.display import clear_output, display

ti.init(arch=ti.gpu)

res = 512
camera = SphereCamera(render_res=res, samples=4)

def run_simulation(duration_frames=30):
    for f in range(duration_frames):
        t = f * 0.1
        mass = 1.0 + math.sin(t * 0.5) * 1.5
        if mass < 0.1: mass = 0.05
        
        camera.geo_engine.update_black_hole_mass(mass)
        camera.cam_pan = t * 0.2
        
        camera.render_black_hole(t)
        img = camera.get_image_data()
        
        clear_output(wait=True)
        plt.figure(figsize=(10,10))
        plt.imshow(img)
        plt.title(f"Frame {f} | Mass: {mass:.2f} Solar Masses")
        plt.axis("off")
        plt.show()

run_simulation(60)

## Mass Scale Guide
- **0.05 - 0.2**: Micro Black Hole (Blue/White, High Hawking Radiation)
- **1.0 - 3.0**: Stellar Mass (Orange Accretion Disk)
- **5.0+**: Supermassive (Red/Deep Orange Disk)